In [ ]:
# conexion al Google Drive
from google.colab import drive
drive.mount('/content/.drive')
!mkdir -p "/content/.drive/My Drive/DMA"
!mkdir -p "/content/buckets"
!ln -s "/content/.drive/My Drive/DMA" /content/buckets/b1

In [ ]:
# instalo  itables solo si no esta instalado
!pip show itables >/dev/null || pip install itables

In [ ]:
import polars as pl
import numpy as np
import math
import matplotlib.pyplot as plt
%matplotlib inline
from IPython import display
import time
import os
import pickle
from functools import reduce
from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)

### Clase de gráficos
Adaptación mínima: si el input tiene más de 2 dimensiones (nuestro caso: 80 componentes), se imprime epoch y MSE por consola en vez de graficar rectas de decisión (que solo tienen sentido en 2D).

In [ ]:
# definicion de la clase de graficos
# ADAPTACION: se agrega guard para datos >2D (80 componentes PCA/ISOMAP)

class perceptron_plot:
    """plotting first hidden layer class"""
    def __init__(self, X, Y, delay) -> None:
        self.X = X
        self.Y = Y
        self.delay = delay
        self.es_2d = (X.shape[1] == 2)

        if not self.es_2d:
            return  # no inicializar gráfico si >2D

        x1_min = np.min(X[:,0])
        x2_min = np.min(X[:,1])
        x1_max = np.max(X[:,0])
        x2_max = np.max(X[:,1])
        self.x1_min = x1_min - 0.1*(x1_max - x1_min)
        self.x1_max = x1_max + 0.1*(x1_max - x1_min)
        self.x2_min = x2_min - 0.1*(x2_max - x2_min)
        self.x2_max = x2_max + 0.1*(x2_max - x2_min)
        self.fig = plt.figure(figsize = (10,8))
        self.ax = self.fig.subplots()
        self.ax.set_xlim(self.x1_min, self.x1_max, auto=False)
        self.ax.set_ylim(self.x2_min, self.x2_max, auto=False)

    def graficarVarias(self, W, x0, epoch, error) -> None:
        if not self.es_2d:
            display.clear_output(wait=True)
            print(f"epoch {epoch:>6d}  |  MSE {error:.6f}")
            return

        display.clear_output(wait =True)
        plt.cla()
        self.ax.set_xlim(self.x1_min, self.x1_max)
        self.ax.set_ylim(self.x2_min, self.x2_max)
        plt.title( 'epoch ' + str(epoch) + '  reg ' + "{0:.2E}".format(error))
        scatter = self.ax.scatter(self.X[:,0], self.X[:,1], c=self.Y, s=20)
        for i in range(len(x0)):
            vx2_min = -(W[i,0]*self.x1_min + x0[i])/W[i,1]
            vx2_max = -(W[i,0]*self.x1_max + x0[i])/W[i,1]
            self.ax.plot([self.x1_min, self.x1_max],
                         [vx2_min, vx2_max],
                         linewidth = 2,
                         color = 'red',
                         alpha = 0.5)
        display.display(plt.gcf())
        time.sleep(self.delay)

In [ ]:
# definicion de las funciones de activacion
#  y sus derivadas
#  ahora agregando las versiones VECTORIZADAS

def func_eval(fname, x):
    match fname:
        case "purelin":
            y = x
        case "logsig":
            y = 1.0 / ( 1.0 + math.exp(-x) )
        case "tansig":
            y = 2.0 / ( 1.0 + math.exp(-2.0*x) ) - 1.0
    return y

# version vectorizada de func_eval
func_eval_vec = np.vectorize(func_eval)


def deriv_eval(fname, y):  #atencion que y es la entrada y=f( x )
    match fname:
        case "purelin":
            d = 1.0
        case "logsig":
            d = y*(1.0-y)
        case "tansig":
            d = 1.0 - y*y
    return d


# version vectorizada de deriv_eval
deriv_eval_vec = np.vectorize(deriv_eval)

### Clase multiperceptron
entrenar, predecir

**Fix mínimo en `predecir()`:** el original usaba variables `epoch` y `MSE` que no existían en ese scope → se reemplazan por `self.red['work']['epoch']` y `self.red['work']['MSE']`. También se salta el gráfico si >2D.

In [ ]:
# definicion de la clase de multiperceptron

class multiperceptron(object):
    """Multiperceptron class"""

    # inicializacion de los pesos de todas las capas
    def _red_init(self, semilla) -> None:
        niveles = self.red['arq']['layers_qty']

        np.random.seed(semilla)
        for i in range(niveles):
           nivel = dict()
           nivel['id'] = i
           nivel['last'] = (i==(niveles-1))
           nivel['size'] = self.red["arq"]["layers_size"][i]
           nivel['func'] = self.red["arq"]["layers_func"][i]

           if( i==0 ):
              entrada_size = self.red['arq']['input_size']
           else:
              entrada_size =  self.red['arq']['layers_size'][i-1]

           salida_size =  nivel['size']

           # los pesos, inicializados random
           nivel['W'] = np.random.uniform(-0.5, 0.5, [salida_size, entrada_size])
           nivel['w0'] = np.random.uniform(-0.5, 0.5, [salida_size, 1])

           # los momentos, inicializados en CERO
           nivel['W_m'] = np.zeros([salida_size, entrada_size])
           nivel['w0_m'] = np.zeros([salida_size, 1])

           self.red['layer'].append(nivel)

    # constructor generico
    def __init__(self) -> None:
        self.data = dict()
        self.red = dict()
        self.carpeta = ""


    # inicializacion full
    def inicializar(self, df, campos, clase, hidden_layers_sizes, layers_func,
                 semilla, carpeta) -> None:

        # genero self.data
        self.data['X'] = np.array( df.select(campos))
        X_mean = self.data['X'].mean(axis=0)
        X_sd = self.data['X'].std(axis=0)
        self.data['X'] = (self.data['X'] - X_mean)/X_sd

        #  Ylabel en  numpy
        label =df.select(clase)
        self.data['Ylabel'] = np.array(label).reshape(len(label))

        # one-hot-encoding de Y
        col_originales = df.columns
        self.data['Y'] = np.array( df.to_dummies(clase).drop(col_originales, strict=False) )

        col_dummies = sorted( list( set(df.to_dummies(clase).columns) -  set(col_originales)))
        clases_originales = reduce(lambda acc, x: acc + [x[(len(clase)+1):]], col_dummies, [])

        tamanos = hidden_layers_sizes
        tamanos.append(self.data['Y'].shape[1])

        arquitectura = {
             'input_size' : self.data['X'].shape[1],
             'input_mean' : X_mean,
             'input_sd' :  X_sd,
             'output_values' : clases_originales,
             'layers_qty' : len(hidden_layers_sizes),
             'layers_size' : tamanos ,
             'layers_func' : layers_func,
        }

        self.red['arq'] = arquitectura


        # inicializo  work
        self.red['work'] = dict()
        self.red['work']['epoch'] = 0
        self.red['work']['MSE'] = float('inf')
        self.red['work']['train_error_rate'] = float('inf')

        self.red['layer'] = list()
        self._red_init(semilla)

        # grabo el entorno
        self.carpeta = carpeta
        os.makedirs(self.carpeta, exist_ok=True)
        with open(self.carpeta+"/data.pkl", 'wb') as f:
            pickle.dump(self.data, f)

        with open(self.carpeta+"/red.pkl", 'wb') as f:
            pickle.dump(self.red, f)


    # Algoritmo Backpropagation
    def  entrenar(self, epoch_limit, MSE_umbral,
               learning_rate, lr_momento, save_frequency,
               retomar=True) -> None:

        # si debo retomar
        if( retomar):
            with open(self.carpeta+"/data.pkl", 'rb') as f:
              self.data = pickle.load(f)

            with open(self.carpeta+"/red.pkl", 'rb') as f:
              self.red = pickle.load(f)


        # inicializaciones del bucle principal del backpropagation
        epoch = self.red['work']['epoch']
        MSE = self.red['work']['MSE']

        # inicializacion del grafico
        grafico = perceptron_plot(X=self.data['X'], Y=self.data['Ylabel'], delay=0.1)

        # continuo mientras error cuadratico medio muy grande  y NO llegué al límite de epochs
        Xfilas = self.data['X'].shape[0]
        niveles = self.red["arq"]["layers_qty"]

        while ( MSE > MSE_umbral) and (epoch < epoch_limit) :
          epoch += 1


          # recorro siempre TODOS los registros de entrada
          for fila in range(Xfilas):
             # fila es el registro actual
             x = self.data['X'][fila:fila+1,:]
             clase = self.data['Y'][fila:fila+1,:]

             # propagar el x hacia adelante, FORWARD
             entrada = x.T  # la entrada a la red

             # etapa forward
             # recorro hacia adelante, nivel a nivel
             vsalida =  [0] *(niveles) # salida de cada nivel de la red

             for i in range(niveles):
               estimulos = self.red['layer'][i]['W'] @ entrada + self.red['layer'][i]['w0']
               vsalida[i] =  func_eval_vec(self.red['layer'][i]['func'], estimulos)
               entrada = vsalida[i]  # para la proxima vuelta


             # etapa backward
             # calculo los errores en la capa hidden y la capa output
             verror =  [0] *(niveles+1) # inicializo dummy
             verror[niveles] = clase.T - vsalida[niveles-1]

             i = niveles-1
             verror[i] = verror[i+1] * deriv_eval_vec(self.red['layer'][i]['func'], vsalida[i])

             for i in reversed(range(niveles-1)):
               verror[i] = deriv_eval_vec(self.red['layer'][i]['func'], vsalida[i])*(self.red['layer'][i+1]['W'].T @ verror[i+1])

             # ya tengo los errores que comete cada capa
             # corregir matrices de pesos, voy hacia atras
             # backpropagation
             entrada = x.T
             for i in range(niveles):
               self.red['layer'][i]['W_m'] = learning_rate *(verror[i] @ entrada.T) + lr_momento *self.red['layer'][i]['W_m']
               self.red['layer'][i]['w0_m'] = learning_rate * verror[i] + lr_momento * self.red['layer'][i]['w0_m']

               self.red['layer'][i]['W']  =  self.red['layer'][i]['W'] + self.red['layer'][i]['W_m']
               self.red['layer'][i]['w0'] =  self.red['layer'][i]['w0'] + self.red['layer'][i]['w0_m']
               entrada = vsalida[i]  # para la proxima vuelta



          # ya recalcule las matrices de pesos
          # ahora avanzo la red, feed-forward
          # para calcular el red(X) = Y
          entrada = self.data['X'].T
          for i in range(niveles):
            estimulos = self.red['layer'][i]['W'] @ entrada + self.red['layer'][i]['w0']
            salida =  func_eval_vec(self.red['layer'][i]['func'], estimulos)
            entrada = salida  # para la proxima vuelta

          # calculo el error cuadratico medio TODOS los X del dataset
          MSE= np.mean( (self.data['Y'].T - salida)**2 )

          # Grafico las rectas SOLAMENTE de la Primera Hidden Layer
          # tengo que hacer w0.T[0]  para que pase el vector limpio
          if( epoch % save_frequency == 0 ) or ( MSE <= MSE_umbral) or (epoch >= epoch_limit) :
              # grafico
              W = self.red['layer'][0]['W']
              w0 = self.red['layer'][0]['w0']
              grafico.graficarVarias(W, w0.T[0], epoch, MSE)

              # almaceno en work
              self.red['work']['epoch'] = epoch
              self.red['work']['MSE'] = MSE
              prediccion = np.argmax( salida.T, axis=1)
              # prediccion
              out = np.array(self.red["arq"]['output_values'])
              error_rate = np.mean( self.data['Ylabel'] != out[prediccion])
              self.red["work"]['train_error_rate'] = error_rate

              # grabo a un archivo la red neuronal  entrenada por donde esté
              #   solo la red, NO los datos
              with open(carpeta+"/red.pkl", 'wb') as f:
                 pickle.dump(self.red, f)

        return (epoch, MSE, self.red['work']['train_error_rate'] )


    # predigo a partir de modelo recien entrenado
    # FIX: epoch y MSE tomados de self.red['work'], y se salta gráfico si >2D
    def  predecir(self, df_new, campos, clase) -> None:
        niveles = self.red['arq']['layers_qty']

        # etapa forward
        # recorro hacia adelante, nivel a nivel
        X_new =  np.array( df_new.select(campos))

        # estandarizo manualmente
        #  con las medias y desvios que almacene durante el entrenamiento
        X_new = (X_new - self.red['arq']['input_mean'])/self.red['arq']['input_sd']

        # grafico los datos nuevos (solo si 2D)
        if X_new.shape[1] == 2:
            Ylabel_new =df_new.select(clase)
            Ylabel_new = np.array(Ylabel_new).reshape(len(Ylabel_new))
            grafico = perceptron_plot(X=X_new, Y=Ylabel_new, delay=0.1)
            W = self.red['layer'][0]['W']
            w0 = self.red['layer'][0]['w0']
            epoch = self.red['work']['epoch']   # FIX
            MSE = self.red['work']['MSE']       # FIX
            grafico.graficarVarias(W, w0.T[0], epoch, MSE)

        # la entrada a la red,  el X que es TODO  x_new
        entrada = X_new.T  # traspongo, necesito vectores columna

        for i in range(niveles):
          estimulos = self.red['layer'][i]['W'] @ entrada + self.red['layer'][i]['w0']
          salida =  func_eval_vec(self.red['layer'][i]['func'], estimulos)
          entrada = salida  # para la proxima vuelta

        # me quedo con la neurona de la ultima capa que se activio con mayor intensidad
        pred_idx = np.argmax( salida.T, axis=1)
        pred_raw = np.max( salida.T, axis=1)

        # calculo error_rate
        out = np.array(self.red['arq']['output_values'])
        error_rate = np.mean( np.array(df_new.select(clase)).flatten() != out[pred_idx])

        return (out[pred_idx], pred_raw, error_rate)


    # cargo un modelo ya entrenado, grabado en carpeta
    def cargar_modelo(self, carpeta) -> None:
        self.carpeta = carpeta

        with open(self.carpeta+"/red.pkl", 'rb') as f:
          self.red = pickle.load(f)

        return (self.red['work']['epoch'],
                self.red['work']['MSE'],
                self.red['work']['train_error_rate'] )

## 1 Lectura del Dataset
En nuestro caso, el dataset no viene de un CSV sino de los arrays `.npy` generados por el notebook de Eigenfaces. Convertimos a DataFrame de Polars con columnas `x1, x2, ..., x80` (los componentes) y `y` (nombre del alumno).

In [ ]:
# ============================================================
# CONFIGURACIÓN — Ajustar estos parámetros según se necesite
# ============================================================
MODO = "pca"    # <<< CAMBIAR A "isomap" PARA LA SEGUNDA RED

CARPETA_DATOS  = "/content/buckets/b1/datos_30x30"
carpeta        = f"/content/buckets/b1/nn_{MODO}/"  # cambiar con cada corrida


In [ ]:
# Lectura del dataset con Polars (adaptado de arrays numpy)

if MODO == "pca":
    X_arr = np.load(f"{CARPETA_DATOS}/fotos_proyectadas_pca.npy")
elif MODO == "isomap":
    X_arr = np.load(f"{CARPETA_DATOS}/fotos_proyectadas_isomap.npy")
else:
    raise ValueError("MODO debe ser 'pca' o 'isomap'")

etiquetas_arr = np.load(f"{CARPETA_DATOS}/etiquetas.npy", allow_pickles=True)

n_componentes = X_arr.shape[1]  # 80
campos = [f"x{i+1}" for i in range(n_componentes)]  # x1, x2, ..., x80

data_dict = {campos[i]: X_arr[:, i].astype(np.float64) for i in range(n_componentes)}
data_dict["y"] = etiquetas_arr.astype(str)

df = pl.DataFrame(data_dict)

print(f"Modo: {MODO}")
print(f"Fotos: {df.shape[0]}  |  Componentes: {n_componentes}")
print(f"Alumnos: {sorted(df['y'].unique().to_list())}")
df

### 1.1 Particion training/testing
Es valido cambiar la semilla_particion para probar distintos (test, train) y asi estimar con mas precisión el error rate en testing (Montecarlo Estimation)

In [ ]:
# particion del dataset en training/testing

semilla_particion = 102191
pct_train = 0.75  # ratio de registros que va a training


def train_test_split_df(df, seed=0, test_size=0.2):
    return df.with_columns(
        pl.int_range(pl.len(), dtype=pl.UInt32)
        .shuffle(seed=seed)
        .gt(pl.len() * test_size)
        .alias("split")
    ).partition_by("split", include_key=False)


(df_train, df_test) = train_test_split_df(df,
                                          seed=semilla_particion,
                                          test_size=pct_train)

# imprimo los tamaños
print("Train:", df_train.shape)
print("Test:", df_test.shape)

# verifico que todas las clases estén en ambos conjuntos
clases_train = set(df_train['y'].unique().to_list())
clases_test  = set(df_test['y'].unique().to_list())
faltantes    = clases_test - clases_train
if faltantes:
    print(f"⚠️  ALERTA: clases en test pero NO en train: {faltantes}")
else:
    print("✓ Todas las clases presentes en ambos conjuntos")

## 2 Entrenamiento del modelo
### 2.1 Inicializacion de la neural network
Es valido cambiar la semilla_red para arrancar el entrenamiento con distintas rectas iniciales

In [ ]:
# defino la red multiperceptron
semilla_red = 200177  # define las rectas iniciales

# Hiperparámetros a experimentar:
hidden = [40, 20]  # capas ocultas (no va la capa final)
n_capas = len(hidden) + 1  # ocultas + salida
layers_func = ['logsig'] * n_capas

mp = multiperceptron()
mp.inicializar(
    df=df_train, campos=campos, clase="y",
    hidden_layers_sizes=hidden.copy(),  # copia porque se muta internamente
    layers_func=layers_func,
    semilla=semilla_red,
    carpeta=carpeta
    )

print(f"Arquitectura: {n_componentes} → {hidden} → {len(clases_train)}")

### 2.2 Entrenamiento de la neural network = backpropagation
Aqui se hace el trabajo pesado de entrenar la red neuronal

Es necesario experimentar con
- learning_rate
- lr_momento
- epoch_limit y MSE_umbral

In [ ]:
# entreno la neural netowork con BackPropagation

# el entrenamiento
(epoch, MSE, train_error_rate) = mp.entrenar(
    epoch_limit=3000,
    MSE_umbral=0.005,
    learning_rate=0.1,
    lr_momento=0.1,
    save_frequency=200,
    retomar=True)

Visualizacion de los resultados de la salida del entrenamiento de la red

In [ ]:
# las metricas basica de la red
print("epoch :", epoch)
print("MSE :", MSE)
print("train_error_rate :", train_error_rate)

### 2.3 Entrenamiento en caso de retomar
Si se cortó el colab, si quiero extender la corrida a mas epochs, si quiero cambiar el learning_rate o el MSE_umbral

In [ ]:
(epoch, MSE, train_error_rate) = mp.entrenar(
    epoch_limit=5000,    # aumento
    MSE_umbral=0.001,
    learning_rate=0.05,
    lr_momento=0.05,
    save_frequency=200,
    retomar=True)

In [ ]:
print("epoch :", epoch)
print("MSE :", MSE)
print("train_error_rate :", train_error_rate)

## 3 Prediccion en los datos de Testing
Se muestran los datos de testing, que son distintos a los de training

**REGLA DE ORO:** La ÚNICA métrica válida para evaluar el modelo y tomar decisiones es el **test error rate**. El train error rate solo sirve para diagnóstico (detectar overfitting). NUNCA tomar decisiones basadas en el train error rate.

### 3.1 Prediccion en caliente

In [ ]:
# aplico la red entrenada al dataset de testing

(y_hat, y_raw, test_error_rate) = mp.predecir(df_new=df_test, campos=campos, clase='y')

Visualizacion del error en testing

In [ ]:
# RECORDAR: la UNICA metrica valida para decidir es test_error_rate
# Si test_error_rate >> train_error_rate → OVERFITTING
print("error_rate (train, test): ",  train_error_rate, test_error_rate)

Visualizacion de la prediccion en testing

In [ ]:
tb_salida_test = pl.DataFrame( {"clase":df_test["y"], "pred":y_hat, "y_raw":y_raw })
tb_salida_test

## 4 Prediccion en datos NUEVOS
La red fue entrenada en el pasado, y se grabó al drive.
Ya no esta disponible la sesion donde se entreno.
No quiero volver a entrenar de cero.

### 4.1 Preparar las fotos nuevas
Las fotos de `caras_test/` deben pasar por el mismo pipeline de Eigenfaces:
1. Detectar rostro (DeepFace/Haar Cascade)
2. Recorte oval, escala de grises, 30×30
3. Aplanar a vector de 900 dimensiones
4. Proyectar con `pca_model.pkl` o `isomap_model.pkl`

In [ ]:
# cargo datos NUEVOS
# NOTA: descomentar cuando las fotos de caras_test/ estén procesadas
#       y proyectadas a los 80 componentes PCA/ISOMAP

# X_new = np.load(f"{CARPETA_DATOS}/fotos_test_proyectadas_{MODO}.npy")
# y_new = np.load(f"{CARPETA_DATOS}/etiquetas_test.npy", allow_pickles=True)
#
# data_new = {campos[i]: X_new[:, i].astype(np.float64) for i in range(n_componentes)}
# data_new["y"] = y_new.astype(str)
# df_new = pl.DataFrame(data_new)
# df_new.shape

In [ ]:
# cargo modelo grabado y lo aplico a los datos nuevos

mp_frio = multiperceptron()
(epoch, MSE, train_error_rate) = mp_frio.cargar_modelo(carpeta)

# (y_hat, y_raw, new_error_rate) = mp_frio.predecir(df_new=df_new, campos=campos, clase='y')

Visualizacion del error modelo aplicado a datos nuevos

In [ ]:
# print("error_rate (train, new): ",  train_error_rate, new_error_rate)

Visualizacion de la prediccion en datos nuevos

In [ ]:
# tb_salida_new = pl.DataFrame( {"clase":df_new["y"], "pred":y_hat, "y_raw":y_raw })
# tb_salida_new

## 5 Umbral de confianza — clasificar como "desconocido"
Si el score máximo de la neurona de salida es menor a un umbral, la red no está segura de la identidad → clasificar como **"desconocido"**.

Esto se usa cuando el profesor evalúe con fotos de personas que no son del grupo.

In [ ]:
UMBRAL_DESCONOCIDO = 0.85  # experimentar con este valor

# Aplicar a los datos nuevos (descomentar cuando estén listos)
# y_hat_final = np.where(
#     y_raw >= UMBRAL_DESCONOCIDO,
#     y_hat,                       # confianza alta → mantener prediccion
#     "desconocido"                # confianza baja → desconocido
# )
#
# tb_final = pl.DataFrame({
#     "clase": df_new["y"],
#     "pred": y_hat_final,
#     "score": np.round(y_raw, 4)
# })
# tb_final

### 5.1 Demo del umbral sobre datos de test
Para verificar que funciona antes de tener las fotos nuevas.

In [ ]:
# demo: aplico el umbral sobre las predicciones de test
y_hat_demo = np.where(
    y_raw >= UMBRAL_DESCONOCIDO,
    y_hat,
    "desconocido"
)

n_desconocidos = np.sum(y_hat_demo == "desconocido")
print(f"Umbral: {UMBRAL_DESCONOCIDO}")
print(f"Fotos clasificadas como 'desconocido': {n_desconocidos} de {len(y_hat_demo)}")

tb_demo = pl.DataFrame({
    "clase": df_test["y"],
    "pred_final": y_hat_demo,
    "score": np.round(y_raw, 4)
})
tb_demo